# Red Teaming at Scale (1/3) · garak — breadth

**Lloyds · AI Agent Security & Governance · Module 7 Day 2 · Dr. Deepali Salwan**

garak is the **broad scanner** — the "nmap for a language model". Its scale is **breadth**:
you point it at a model and it fires *many* known attack probes across *several* weakness
families at once, then reports a pass rate per probe.

> Runs in **Google Colab**. Paste your OpenAI key. We cap the run so it finishes in a few
> minutes and costs a few cents — then show how to open it up to a full scan.


In [ ]:
# install garak
!pip install -q -U garak

In [ ]:
# garak reads the key from the OPENAI_API_KEY environment variable
import os
from getpass import getpass
os.environ["OPENAI_API_KEY"] = getpass("Paste your OpenAI API key (sk-...): ")
print("Key set.")

## A broad scan across several families

This is the "at scale" part: instead of one probe, we fire **two whole families** — `dan`
(jailbreaks) and `promptinject` (prompt injection) — which is *dozens* of individual attacks.
`--generations 2` caps attempts per probe so the first run stays quick and cheap.

- `--probes dan,promptinject` → two families (many probes)
- `--generations 2` → 2 attempts each (raise for a harder test)

*(Newer garak also accepts `--target_type` / `--target_name`. If a flag errors, run
`!python -m garak --help` to see your version's names.)*


In [ ]:
!python -m garak --model_type openai --model_name gpt-4o-mini \
    --probes dan,promptinject --generations 2

## Read the result — coverage + pass rates

garak prints a line **per probe** with a pass rate (the inverse of Attack Success Rate): high =
the model resisted, low = the jailbreak often worked. Because we ran *two families*, you get a
**breadth** picture in one run — exactly what a broad scanner is for.

It also writes an **HTML report** + a `.jsonl` log (path printed at the end of the run).


In [ ]:
# find the report garak just wrote
!ls -t ~/.local/share/garak/garak_runs/ 2>/dev/null | head -5 || ls -t *.jsonl *.html 2>/dev/null | head -5

## Scale it up (optional)

- **More families:**  `--probes dan,promptinject,encoding,leakreplay`
- **Everything (slow, costs more):**  `--probes all`
- **List the whole catalogue:**  `!python -m garak --list_probes`

### Where garak fits in "at scale"
garak gives you **breadth, cheaply** — a fast first pass over the known-attack catalogue on every
model or version. It finds the *known* weaknesses so your humans (and PyRIT) can hunt the novel ones.
